1. **preprocessing**

In [ ]:
import pandas as pd
import numpy as np

movies = pd.read_csv("tmdb_5000_movies.csv")
credits = pd.read_csv("tmdb_5000_credits.csv")

In [ ]:
df = movies.merge(credits, on="title")

In [ ]:
print(df.shape)
df.head()
df.info()

(4809, 23)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4809 entries, 0 to 4808
Data columns (total 23 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   budget                4809 non-null   int64  
 1   genres                4809 non-null   object 
 2   homepage              1713 non-null   object 
 3   id                    4809 non-null   int64  
 4   keywords              4809 non-null   object 
 5   original_language     4809 non-null   object 
 6   original_title        4809 non-null   object 
 7   overview              4806 non-null   object 
 8   popularity            4809 non-null   float64
 9   production_companies  4809 non-null   object 
 10  production_countries  4809 non-null   object 
 11  release_date          4808 non-null   object 
 12  revenue               4809 non-null   int64  
 13  runtime               4807 non-null   float64
 14  spoken_languages      4809 non-null   object 
 15  status    

**Converting from json to python list**

In [ ]:
import ast

def extract_names(text):
    try:
        items = ast.literal_eval(text)
        return [i['name'] for i in items]
    except:
        return []

df['genres'] = df['genres'].apply(extract_names)
df['main_genre'] = df['genres'].apply(lambda x: x[0] if x else "Unknown")

In [ ]:
def extract_company(text):
    try:
        items = ast.literal_eval(text)
        return items[0]['name'] if len(items) > 0 else "Unknown"
    except:
        return "Unknown"

df['main_company'] = df['production_companies'].apply(extract_company)

**Data Cleaning & Feature Engineering**

In [ ]:
df['release_date'] = pd.to_datetime(df['release_date'], errors='coerce')
df['release_year'] = df['release_date'].dt.year

In [ ]:
df = df.dropna(subset=['budget', 'revenue', 'runtime'])

In [ ]:
df["popularity"] = df["popularity"].fillna(0)

In [ ]:
df['original_language'] = df['original_language'].fillna("Unknown")

In [ ]:
df['profit'] = df['revenue'] - df['budget']

In [ ]:
df = df[df['budget'] > 0]
df = df[df['revenue'] > 0]
df = df[df['runtime'] > 0]

In [ ]:
df['profit_category'] = df['profit'].apply(
    lambda x: "High Profit" if x > 100000000
    else "Medium Profit" if x > 0
    else "Loss"
)

In [ ]:
df['budget_category'] = df['budget'].apply(
    lambda x: "High Budget" if x > 100000000
    else "Medium Budget" if x > 30000000
    else "Low Budget"
)

In [ ]:
df['roi'] = df['revenue'] / df['budget']
df['roi'] = df['roi'].replace([float('inf'), -float('inf')], 0)

Filtering Top 10 Movies

In [ ]:
top_genres = (
    df.groupby("main_genre")["profit"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .index
)

filtered_df = df[df["main_genre"].isin(top_genres)]

Column other instead of removing

In [ ]:
df["main_genre_grouped"] = df["main_genre"].apply(
    lambda x: x if x in top_genres else "Other"
)

In [ ]:
filtered_df = df[df["release_year"] >= 2010]

Aggregations

In [ ]:
grouped = df.groupby(["main_genre", "profit_category"])["profit"].sum().reset_index()

In [ ]:
df.dtypes

,0
budget,int64
genres,object
homepage,object
id,int64
keywords,object
original_language,object
original_title,object
overview,object
popularity,float64
production_companies,object


In [ ]:
df = df[[
    'title',
    'main_genre',
    'main_company',
    'budget',
    'revenue',
    'profit',
    'runtime',
    'original_language',
    'vote_average',
    'release_date'
    ,'popularity'
    ,'profit_category'
    ,'budget_category'
    ,'roi',
    'main_genre_grouped'
]]

In [ ]:
df.to_csv("cleaned_data.csv", index=False)